# A8 calibration and locked evaluation
Run calibration before the locked test is uploaded or copied into the runtime. All logic is in `sipature_ml.evaluation`; this notebook only orchestrates the two immutable phases.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/hackathon/ml
!python -m pip uninstall -y torchvision
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .

## Phase 1: validation only
Confirm that the runtime split directory contains the manifest and validation file but not the test file. The model run must be the full A7 Drive run, including `manifest.json` and both model directories.

In [ ]:
from pathlib import Path
from sipature_ml.evaluation import run_calibration

SPLIT_DIR = Path('/content/a8-splits')
MODEL_RUN_DIR = Path('/content/drive/MyDrive/SIPATURE/runs/REPLACE_A7_RUN_ID')
CALIBRATION_DIR = Path('/content/drive/MyDrive/SIPATURE/calibration/REPLACE_CALIBRATION_ID')
calibration = run_calibration(SPLIT_DIR, MODEL_RUN_DIR, CALIBRATION_DIR)
assert calibration['test_read'] is False
calibration

## Mandatory pause
Stop here. Inspect `calibration.json` and `manifest.json`, record their hashes, and freeze the directory. Do not continue until a human confirms the artifact is accepted. Only after that confirmation may the locked test file be uploaded or copied to `SPLIT_DIR`.

In [ ]:
confirmation = input('Type FROZEN to confirm calibration inspection and authorization: ')
assert confirmation == 'FROZEN', 'Locked test access is not authorized'

## Phase 2: one locked-test pass
After the pause, copy the hash-locked test file into `SPLIT_DIR`, then execute this cell once. Choose a new output directory; overwrite is refused.

In [ ]:
from sipature_ml.evaluation import run_locked_test_evaluation

EVALUATION_DIR = Path('/content/drive/MyDrive/SIPATURE/evaluation/REPLACE_EVALUATION_ID')
BASELINE_METRICS_DIR = Path('/content/hackathon/ml/artifacts/metrics')
metrics = run_locked_test_evaluation(
    SPLIT_DIR, MODEL_RUN_DIR, CALIBRATION_DIR, EVALUATION_DIR, BASELINE_METRICS_DIR
)
assert metrics['test_inference_passes'] == 1
metrics